In [ ]:
import os
from os import path
from datetime import datetime
import random
import pickle
import pytz
import requests

import pandas as pd
import numpy as np

import networkx as nx

from utils.parsing import load_properties, hour_rounder
from utils.parsing import get_contact_list, get_infection_list, get_node_state, create_contact_network, remove_nodes_with_less_edges

## Initial data loading and parsing

In [ ]:
# Settings for data analysis

dataset_id = 202
data_format = 3

# Discard transmissions when the infected node was already infected before
discard_reinfections = True

# Default contact time for transmissions that are missing an associated contact event
def_contact_time = 1

# Time delta for animation in seconds, needs to be small in comparison with the total length of the 
# simulation so the changes in the animation are smooth
anim_time_step_min = 10
anim_time_delta_sec = anim_time_step_min * 60

# Time delta for plots in some "natural" time step, for examply, hourly or daily
nat_time_step_min = 24 * 60
nat_time_delta_sec = nat_time_step_min * 60

min_total_contact_time = 0.16  # at least this total time (in minutes) over the two weeks to be defined as in contact
min_total_contact_count = 1 # nodes must have at least this number of edges with other nodes to be kept

# Set a random seed for reproducibility
random_seed = 32
random.seed(random_seed)

# Print warning messages to the console when parsing data
print_data_warnings = False

In [ ]:
data_folder = './data'
output_folder = './output'
if not path.exists(output_folder):
    os.makedirs(output_folder)

# Load time properties, which need to be in the file time.properties
# under the data folder with the following structure:
# sim_tz=Europe/London
# time0=Jun 30 2025 8:30AM
# time1=Jun 30 2025 5:30PM
tprops_filename = path.join(data_folder, "time.properties")
tprops = load_properties(tprops_filename)
sim_tz = tprops['sim_tz']
time0 = tprops['time0']
time1 = tprops['time1']

print('Time zone =', sim_tz)
print('Start time =', time0)
print('End time =', time1)

# https://howchoo.com/g/ywi5m2vkodk/working-with-datetime-objects-and-timezones-in-python
# https://itnext.io/working-with-timezone-and-python-using-pytz-library-4931e61e5152
timezone = pytz.timezone(sim_tz)
obs_date0 = timezone.localize(datetime.strptime(time0, '%b %d %Y %I:%M%p'))
obs_date1 = timezone.localize(datetime.strptime(time1, '%b %d %Y %I:%M%p'))

if obs_date0 and obs_date1:
    tmin = datetime.timestamp(obs_date0)
    tmax = datetime.timestamp(obs_date1)
else:
    tmin = min_time
    tmax = max_time

num_nat_intervals = int((nat_time_delta_sec + tmax - tmin) / nat_time_delta_sec + 1)
print("Intervals for time analysis =", num_nat_intervals)

In [ ]:
# Load participants and histories

all_users = pd.read_csv(path.join(data_folder, "participants.csv"), low_memory=False) 
all_events = pd.read_csv(path.join(data_folder, "histories.csv"), low_memory=False)
    
users = all_users[all_users["sim_id"] == dataset_id].copy()
if data_format < 2:
    if data_format < 1:
        users['random_id'] = users['id']
else:
    users['random_id'] = users['random_id'].astype(str).str.zfill(4)

# Save the users to a pickle file
with open(path.join(data_folder, 'users.pickle'), 'wb') as f:
    pickle.dump(users, f)

if data_format == 1:
    events = all_events
else:
    events = all_events[all_events["sim_id"] == dataset_id].copy()
    
events.fillna({'contact_length':0, 'peer_id':-1}, inplace=True)
events["event_start"] = events["time"] - events["contact_length"]/1000
events["event_start"] = events["event_start"].astype(int)

p2pToSim = pd.Series(users.sim_id.values, index=users.p2p_id).to_dict()
p2pToId = pd.Series(users.id.values, index=users.p2p_id).to_dict()
idTop2p = pd.Series(users.p2p_id.values, index=users.id).to_dict()
        
user_index = {}
index_user = {}
idx = 0
for kid in idTop2p:
    user_index[kid] = idx
    index_user[idx] = kid
    idx += 1

# Round min and max times to the hour
min_time = min(events['time'])
max_time = max(events['time'])
first_date = hour_rounder(datetime.fromtimestamp(min_time, tz=timezone))
last_date = hour_rounder(datetime.fromtimestamp(max_time, tz=timezone))
min_time = datetime.timestamp(first_date)
max_time = datetime.timestamp(last_date)

print("First event:", first_date)
print("Last event :", last_date)

if time0 and time1:
    print("Start time:", datetime.strptime(time0, '%b %d %Y %I:%M%p'))
    print("End time:", datetime.strptime(time1, '%b %d %Y %I:%M%p'))

print(first_date.tzinfo)

# These should return the same value
print(len(users))
print(len(idTop2p))    
print(len(p2pToId))
print(len(user_index))

At this point, we have parsed the source simulation data and we can use it to extract any information we need from it. For example, in the cell below, we get the final states of all nodes, the list of infections (all the (infectors, infectees) pairs) and the list of all contacts during the entire simulation:

In [ ]:
# Get list of infections and contacts, needed to construct the network graph

state = get_node_state(user_index, events, None, p2pToId, data_format, print_data_warnings)
infections = get_infection_list(user_index, events, discard_reinfections, anim_time_delta_sec, p2pToId, data_format, print_data_warnings)
contacts = get_contact_list(user_index, events, infections, def_contact_time, p2pToId, data_format, print_data_warnings)

With the contacts and state information, we can build the network graph using the networkx package. The first step is to construct the full network where we remove isolated nodes:

In [ ]:
G = create_contact_network(user_index, contacts, state, "final_health_state", min_total_contact_time)

print(len(G.nodes()), len(G.edges()))

removed = remove_nodes_with_less_edges(G, min_total_contact_count)

isolates = list(nx.isolates(G))
G.remove_nodes_from(isolates)

mask = users.index.isin(removed + isolates)
remids = pd.DataFrame(users[mask]['random_id'].tolist(), columns=['User ID'])
remids.to_csv(path.join(data_folder, 'removed-nodes.csv'), index=False)

print('Removed', len(remids), 'nodes without enough connections')
print('There are', len(G.nodes()), 'remaining nodes with', len(G.edges()), 'edges between them')

In [ ]:
# Construct a new graph using only the transmission (infection) data
T = nx.DiGraph(infections)

with open(path.join(data_folder, 'transmission-tree.pickle'), 'wb') as f:
    pickle.dump(T, f)

In [ ]:
# If the Graph has more than one component, this will return False:
print("Network is connected", nx.is_connected(G))

components = nx.connected_components(G)

subgraphs = [G.subgraph(c) for c in components]
for sg in subgraphs:
    print(len(sg.nodes()), len(sg.edges()))

# Calculate the largest connected component subgraph:
G = sorted(subgraphs, key=lambda x: len(x))[-1]

degrees = [degree for node, degree in G.degree()]

with open(path.join(data_folder, 'network-largest_conn_comp.pickle'), 'wb') as f:
    pickle.dump(G, f)

# Calculating the node positions for the largest connected component
Gpos = nx.spring_layout(G, seed=random_seed)

with open(path.join(data_folder, 'network-node-positions.pickle'), 'wb') as f:
    pickle.dump(Gpos, f)

## Adding additional data to the list of users

In [ ]:
users['group'] = users['group'].map({'group1': 1, 'group2': 2})

In [ ]:
surv_q_id2title = {35:"S1_Q1", 36:"S1_Q2", 37:"S1_Q3", 38:"S1_Q4", 39:"S1_Q5", 40:"S1_Q6",
                   41:"S2_Q1", 42:"S2_Q2", 43:"S2_Q3", 44:"S2_Q4",
                   55:"S3_Q1", 56:"S3_Q2", 57:"S3_Q3", 58:"S3_Q4", 59:"S3_Q5"}

surv_a_lett2num = {'a':1, 'b':2, 'c':3, 'd':4, 'e':5, 'f':6}

In [ ]:
all_survey_answers = pd.read_csv(path.join(data_folder, "survey-answers.csv"), low_memory=False) 

In [ ]:
survey_answers = all_survey_answers[all_survey_answers["sim_id"] == dataset_id]

In [ ]:
qid_values = all_survey_answers.question_id.values
uid_values = all_survey_answers.user_id.values
ans_values = all_survey_answers.value.values

In [ ]:
for k in surv_q_id2title:
    users[surv_q_id2title[k]] = np.nan
users['quarantine'] = np.nan
users['no_quarantine'] = np.nan

In [ ]:
for uid, qid, ans in zip(uid_values, qid_values, ans_values):
    indices = users.index[users['id'] == uid].tolist()
    qcol = surv_q_id2title[qid]
    qval = surv_a_lett2num[ans]    
    for idx in indices:
        users.at[idx, qcol] = qval

In [ ]:
# Remove events before a certain date

# tloc = timezone.localize(datetime.strptime('Nov 19 2025 8:30PM', '%b %d %Y %I:%M%p'))
# tstop = datetime.timestamp(tloc)
# events = events[events['time'] < tstop].copy()

In [ ]:
qevents = events[(events['inf'] == 'quarantine')]
nqevents = events[(events['inf'] == 'noQuarantine')]

for uid in users.id.values:
    indices = users.index[users['id'] == uid].tolist()
    qindices = qevents.index[qevents['user_id'] == uid].tolist()
    nqindices = nqevents.index[nqevents['user_id'] == uid].tolist()    
    for idx in indices:
        users.at[idx, 'quarantine'] = len(qindices)
        users.at[idx, 'no_quarantine'] = len(nqindices)

In [ ]:
# Remove users who did not ever pick quarantine

condition = (users['quarantine'] == 0) & (users['no_quarantine'] == 0)
users = users[~condition].copy()

In [ ]:
users['total_trials'] = users['quarantine'] + users['no_quarantine']
users['quarantine_rate'] = users['quarantine'] / users['total_trials']

In [ ]:
dates = []
infections = []
quarantine_no = []
quarantine_yes = []

t = tmin
day = 0
print('Calculating daily counts (infections, quarantine)...')
while t <= tmax:
    t0 = t
    t += nat_time_delta_sec
    td = datetime.fromtimestamp(t, tz=timezone)
    
    # We want to include events that ocurred between t0 and t
    condition = (t0 < events['time']) & (events['time'] <= t)

    tevents = events[condition]
    itevents = tevents[(tevents['type'] == 'infection')]
    qtevents = tevents[(tevents['inf'] == 'quarantine')]
    nqtevents = tevents[(tevents['inf'] == 'noQuarantine')]

    date = datetime.fromtimestamp(t0, tz=timezone).strftime('%Y-%m-%d')
    
    users[f'quarantine_day{day}'] = np.nan
    users[f'no_quarantine_day{day}'] = np.nan
    for uid in users.id.values:
        indices = users.index[users['id'] == uid].tolist()
        qindices = qtevents.index[qtevents['user_id'] == uid].tolist()
        nqindices = nqtevents.index[nqtevents['user_id'] == uid].tolist()    
        for idx in indices:
            users.at[idx, f'quarantine_day{day}'] = len(qindices)
            users.at[idx, f'no_quarantine_day{day}'] = len(nqindices)

    date = datetime.fromtimestamp(t0, tz=timezone).strftime('%Y-%m-%d')
    qno = len(nqtevents)
    qyes = len(qtevents)
    inf = len(itevents)

    print(f'day {day}', date, qno, qyes, inf)

    dates.append(date)
    infections.append(inf)
    quarantine_no.append(qno)
    quarantine_yes.append(qyes)    
    
    day += 1
print('Done')

daily_counts = pd.DataFrame({'date': dates, 
                             'infections': infections,
                             'quarantine_no': quarantine_no, 
                             'quarantine_yes': quarantine_yes})

In [ ]:
# Make sure that all participants have at least a score of 1
users.loc[users['score'] <= 0, 'score'] = 1

In [ ]:
users

In [ ]:
with open(path.join(data_folder, 'users.pickle'), 'wb') as f:
    pickle.dump(users, f)

with open(path.join(data_folder, 'daily_counts.pickle'), 'wb') as f:
    pickle.dump(daily_counts, f)

## Data summaries

In [ ]:
infections = events[events["type"] == "infection"]
ilist = {}
for uid, t in zip(infections.user_id.values, infections.time.values):
    ilist[uid] = t

illness = events[events["type"] == "illness"]
slist = {}
for uid, t, i in zip(illness.user_id.values, illness.time.values, illness.inf.values):
    if i == 'symptomatic':
        slist[uid] = t

devices = events.query('type == "modifier" and modifier.str.contains("model:")')
dlist = {}
ios_count = 0
android_count = 0
for uid, dev in zip(devices.user_id.values, devices.modifier.values):
    dlist[uid] = dev
    if 'iPhone' in dev or 'iPad' in dev:
        ios_count += 1
    else:
        android_count += 1

outcomes = events[events["type"] == "outcome"]
elist = {}
qlist = {}
escaped_on_ios = 0
escaped_on_android = 0
for uid, t, o in zip(outcomes.user_id.values, outcomes.time.values, outcomes.out.values):
    if o == 'ESCAPED':        
        elist[uid] = t
        if uid in dlist:
            if 'iPhone' in dlist[uid] or 'iPad' in dlist[uid]:
                escaped_on_ios += 1
            else:
                escaped_on_android += 1
    if o == 'QUIT':
        qlist[uid] = t

for uid in ilist:
    if uid in elist and ilist[uid] < elist[uid]:
        print(uid, 'escaped after becoming infected')
    if uid in qlist and ilist[uid] < qlist[uid]:
        print(uid, 'quit after becoming infected')

print('Participants using iOS devices =', ios_count)
print('Participants using Android devices =', android_count)
print('Infected =', len(ilist))
print('Symptomatic =', len(slist))
print('Escaped =', len(elist))
print('Escaped using iOS device =', escaped_on_ios)
print('Escaped using Android device =', escaped_on_android)
print('Quit =', len(qlist)) 